# Infer — How Does Software Know What to Ask Next?
### Pune AI Builders Meetup — Notebook 02

---

> **Where we left off:** We parsed Arjun's resume and got a structured profile.
> The LLM flagged: salary, notice period, and career gaps are missing.
>
> **New question:** *Which of those should we ask first — and how does software decide?*

**Inference** = reasoning about what you still don't know, and deciding what matters most.

In [2]:
OPENAI_API_KEY="sk-proj-****************"  # ← replace with your OpenAI API key

In [3]:
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY))

# ── Arjun's profile as extracted by the LLM in Notebook 01 ─────────────────
# Fields the LLM could read from the resume are filled in.
# Fields that require asking the candidate are None.

ARJUN = {
    # ── From resume ─────────────────────────────────────────────────────────
    "name"              : "Arjun Mehta",
    "current_role"      : "Senior SDE-II, Payments Infra",
    "current_company"   : "Flipkart",
    "total_experience"  : 8,
    "top_skills"        : ["Java", "Go", "distributed systems", "Kafka",
                           "Redis", "MySQL/Postgres", "system design", "Python"],
    "certifications"    : ["AWS Solutions Architect Associate"],
    "career_goal"       : "Senior IC or Tech Lead role in fintech, payments, or infrastructure",
    "one_line_summary"  : "Senior Backend Engineer with expertise in fintech and distributed systems",

    "companies_timeline": [
        {"company": "Wipro Technologies", "role": "Software Engineer",
         "from": "Aug 2015", "to": "Jul 2017",  "duration_months": 24},
        {"company": "Ola Cabs",           "role": "Senior SDE, Risk & Trust",
         "from": "Mar 2018", "to": "Feb 2021", "duration_months": 36},
        {"company": "Flipkart",           "role": "Senior SDE-II, Payments Infra",
         "from": "Sep 2021", "to": "present",  "duration_months": 25},
    ],

    # Two silent gaps — no reason given in resume for either
    "career_gaps": [
        {"from": "Jul 2017", "to": "Mar 2018", "duration_months": 8,
         "reason_given": False, "reason": None},
        {"from": "Feb 2021", "to": "Sep 2021", "duration_months": 7,
         "reason_given": False, "reason": None},
    ],

    "notable_projects": [
        {"name": "FraudNet",    "company": "Ola",          "year": "2019–2020",
         "tech_used": ["Java", "Kafka Streams", "Redis", "Drools", "Python"],
         "impact": "₹2Cr fraud prevented in first quarter post-launch."},
        {"name": "PayRecon",    "company": "Flipkart",     "year": "2022–ongoing",
         "tech_used": ["Go", "gRPC", "Postgres", "AWS SQS"],
         "impact": "SLA: 99.99% accuracy across 12 payment gateways."},
        {"name": "review-gpt",  "company": "personal OSS", "year": "2021",
         "tech_used": ["Python", "GPT-4"],
         "impact": "~340 GitHub stars. Used by 12+ companies."},
    ],

    # ── Must be discovered through conversation (None = not yet known) ───────
    "salary_expectation": None,   # not in resume
    "notice_period"     : None,   # not in resume
    "reason_for_change" : None,   # not in resume
    "open_to_relocation": None,   # not in resume
}

print("Setup done.")
print(f"\nArjun's profile — known fields: "
      f"{sum(1 for v in ARJUN.values() if v is not None)} / {len(ARJUN)}")
print(f"Career gaps to explain: {len(ARJUN['career_gaps'])}")
print(f"Skills: {ARJUN['top_skills']}")
print(f"Still need to ask: {[k for k, v in ARJUN.items() if v is None]}")


Setup done.

Arjun's profile — known fields: 11 / 15
Career gaps to explain: 2
Skills: ['Java', 'Go', 'distributed systems', 'Kafka', 'Redis', 'MySQL/Postgres', 'system design', 'Python']
Still need to ask: ['salary_expectation', 'notice_period', 'reason_for_change', 'open_to_relocation']


---
## Approach 1 — Traditional: Hardcoded Question List

For every missing field → one fixed question. Ask all of them. Same order. Every candidate.

In [4]:
FIXED_QUESTIONS = {
    "salary_expectation": "What is your salary expectation?",
    "notice_period"     : "What is your notice period?",
    "reason_for_change" : "Why are you looking for a change?",
    "gap_explanation"   : "Can you explain any gaps in your employment?",
    "open_to_relocation": "Are you open to relocation?",
}

missing = [(f, q) for f, q in FIXED_QUESTIONS.items() if not ARJUN.get(f)]

print(f"Traditional system asks all {len(missing)} missing fields, in this fixed order:\n")
for i, (field, question) in enumerate(missing, 1):
    print(f"  Q{i}. {question}")

Traditional system asks all 5 missing fields, in this fixed order:

  Q1. What is your salary expectation?
  Q2. What is your notice period?
  Q3. Why are you looking for a change?
  Q4. Can you explain any gaps in your employment?
  Q5. Are you open to relocation?


**The problem:** It asks `gap_explanation` to every candidate, even those with no gaps.
It doesn't know which question is most important *for this specific candidate, right now*.

**It's a form**. Not a conversation.

---
## Approach 2 — LLM Inference: What's the Most Important Question Right Now?

In [6]:
def infer_next_question(profile: dict, history: list[dict]) -> dict:
    """Ask the LLM: given what we know, what ONE question should we ask next?"""

    past = "\n".join(
        f"  {'Counsellor' if m['role'] == 'assistant' else 'Candidate'}: {m['content']}"
        for m in history
    ) or "  None yet."

    # Summarise unexplained gaps so the LLM can reason about them specifically
    unexplained_gaps = [
        f"{g['from']} → {g['to']} ({g['duration_months']} months)"
        for g in profile.get("career_gaps", [])
        if not g.get("reason_given")
    ]

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are a career counsellor doing candidate discovery.
To give good advice you need: salary_expectation, notice_period, reason_for_change,
open_to_relocation. If career_gaps contains unexplained gaps, those take top priority.

Return ONLY JSON:
{
  "has_enough_info": bool,
  "most_important_gap": "field name — one of: salary_expectation, notice_period, reason_for_change, open_to_relocation, or gap_<from>_<to> for a specific career gap",
  "reasoning": "one sentence why this field matters most right now",
  "next_question": "one natural conversational question — not a form prompt"
}
Rules:
- ONE question only.
- Unexplained career gaps take priority over logistics (salary, notice).
- When multiple gaps exist, address them one at a time (earliest gap first).
- Set has_enough_info=true only when all gaps are explained AND salary + notice known."""
            },
            {
                "role": "user",
                "content": (
                    f"Profile:\n{json.dumps(profile, indent=2)}\n\n"
                    f"Unexplained career gaps: {unexplained_gaps or 'none'}\n\n"
                    f"Conversation so far:\n{past}"
                )
            }
        ],
        temperature=0.3
    )

    raw = resp.choices[0].message.content.strip()
    return json.loads(raw)


# Ask the LLM what to ask Arjun first
result = infer_next_question(ARJUN, [])

print(f"Most important gap : {result['most_important_gap']}")
print(f"Why               : {result['reasoning']}")
print(f"\nQuestion to ask   : \"{result['next_question']}\"")


Most important gap : gap_2017-07_2018-03
Why               : Understanding the reason for the 8-month gap is crucial to assess your career trajectory and commitment.

Question to ask   : "Can you share what you were doing during the gap between July 2017 and March 2018?"


---
## Key Demo — Same System, Three Different Candidates

The traditional system asks the same questions in the same order.
Watch what the LLM does when the profile is different.

In [7]:
candidates = [
    {
        "name": "Arjun Mehta",
        "situation": "Strong profile, two silent timeline gaps, logistics unknown",
        "profile": {
            "current_role": "Senior SDE-II, Payments Infra", "current_company": "Flipkart",
            "total_experience": 8,
            "top_skills": ["Java", "Go", "distributed systems", "Kafka", "Redis",
                           "MySQL/Postgres", "system design", "Python"],
            "certifications": ["AWS Solutions Architect Associate"],
            "career_goal": "Senior IC or Tech Lead in fintech/payments/infra",
            "career_gaps": [
                {"from": "Jul 2017", "to": "Mar 2018", "duration_months": 8,
                 "reason_given": False, "reason": None},
                {"from": "Feb 2021", "to": "Sep 2021", "duration_months": 7,
                 "reason_given": False, "reason": None},
            ],
            "salary_expectation": None, "notice_period": None,
            "reason_for_change": None,  "open_to_relocation": None,
        }
    },
    {
        "name": "Priya Nair",
        "situation": "No gaps, logistics known — but career goal is completely unclear",
        "profile": {
            "current_role": "SDE-II at TCS", "total_experience": 4,
            "top_skills": ["Java", "Spring Boot", "MySQL"],
            "career_goal": None,
            "career_gaps": [],   # no gaps
            "salary_expectation": None, "notice_period": "90 days",
            "reason_for_change": "wants growth", "open_to_relocation": None,
        }
    }
]

for c in candidates:
    r = infer_next_question(c["profile"], [])
    gaps = c["profile"].get("career_gaps", [])
    print(f"  {c['name']:<14} | {c['situation']}")
    print(f"  {'':14} | Gaps: {len(gaps)} unexplained")
    print(f"  {'':14} | LLM asks about: {r['most_important_gap']}")
    print(f"  {'':14} | Question: \"{r['next_question']}\"")
    print()

print("Traditional system: same 5 questions, same order, for both.")


  Arjun Mehta    | Strong profile, two silent timeline gaps, logistics unknown
                 | Gaps: 2 unexplained
                 | LLM asks about: gap_2017_07_2018_03
                 | Question: "Can you share what you were doing during the gap from July 2017 to March 2018?"

  Priya Nair     | No gaps, logistics known — but career goal is completely unclear
                 | Gaps: 0 unexplained
                 | LLM asks about: salary_expectation
                 | Question: "What are your salary expectations for your next role?"

Traditional system: same 5 questions, same order, for both.


---
## Conversation Loop — Ask → Learn → Decide → Stop

The LLM asks one question. We give it an answer. It decides whether to keep going or stop.

In [8]:
import copy

profile = copy.deepcopy(ARJUN)
history = []

print("=== Conversation Loop — type answers as the candidate ===\n")

for turn in range(1, 8):
    r = infer_next_question(profile, history)

    if r["has_enough_info"]:
        print(f"\nTurn {turn}: LLM has enough — stopping.")
        break

    field    = r["most_important_gap"]
    question = r["next_question"]

    print(f"Turn {turn} [{field}]")
    print(f"  Counsellor: {question}")
    answer = input("  Candidate : ").strip() or "No answer."
    print()

    history.append({"role": "assistant", "content": question})
    history.append({"role": "user",      "content": answer})

    # Gap fields update the career_gaps entry; other fields set top-level keys
    if field.startswith("gap_"):
        for gap in profile["career_gaps"]:
            if not gap["reason_given"]:
                gap["reason_given"] = True
                gap["reason"]       = answer
                break
    else:
        profile[field] = answer

print("\nProfile after conversation:")
for gap in profile["career_gaps"]:
    icon = "✓" if gap["reason_given"] else "✗"
    print(f"  {icon}  gap {gap['from']} → {gap['to']}: {gap['reason'] or 'unexplained'}")
for k in ["salary_expectation", "notice_period", "reason_for_change", "open_to_relocation"]:
    v = profile.get(k)
    print(f"  {'✓' if v else '✗'}  {k:<24}: {v or 'unknown'}")


=== Conversation Loop — type answers as the candidate ===

Turn 1 [gap_2017_03_2018]
  Counsellor: Can you share what you were doing during the gap between July 2017 and March 2018?

Turn 2 [gap_2021_02_2021_09]
  Counsellor: What was the reason for your gap between February and September 2021?

Turn 3 [salary_expectation]
  Counsellor: What are your salary expectations for your next role?

Turn 4 [notice_period]
  Counsellor: What is your current notice period at Flipkart?

Turn 5 [reason_for_change]
  Counsellor: What is prompting you to look for a new role at this time?


Turn 6: LLM has enough — stopping.

Profile after conversation:
  ✓  gap Jul 2017 → Mar 2018: I was on medicle leave
  ✓  gap Feb 2021 → Sep 2021: same
  ✓  salary_expectation      : 80LPA
  ✓  notice_period           : 2 months
  ✓  reason_for_change       : better role
  ✗  open_to_relocation      : unknown


---
## The Aha Moment

```
Traditional:  fixed list → ask all missing fields → done
LLM:          reason → ask what matters most → stop when ready
```

The key difference: `ANSWERS.get(field)` — we use the LLM's own decision (`most_important_gap`)
to look up the answer. No keyword matching. No hardcoded order. The LLM chose the field.

| | Traditional | LLM Inference |
|---|---|---|
| Question order | Fixed | Based on what matters most right now |
| Asks things it already knows? | Yes | No |
| Asks `gap_explanation` if no gaps? | Yes | No |
| Stopping condition | Form complete | Has enough to give good advice |

---

> **Understand → *Infer* → Plan → Act → Explain**

---
## What's Next?

We have a complete profile. Now we need to *act* on it — search jobs, check GitHub, make recommendations.
But the LLM can't do any of that on its own. It has no internet, no database, no tools.

→ **Notebook 03: Act — Tool Calling**